# What Should a Latent Message Preserve?**DAPLab Latent-comm task.** Songhan (Mason) Wu · [github.com/SonghanWu](https://github.com/SonghanWu)---### The argument in one paragraphEvery KV-cache compression method in use today ranks tokens by a statistic computed**inside the sender**: accumulated attention (H2O), the attention a trailing observationwindow pays to the prefix (SnapKV), or position (StreamingLLM). That is the rightquestion to ask when a model is talking to itself, because then there is no receiver.An agent handoff is different: the receiver is a separate instance with its own context.Classical distributed source coding says the sender's own statistics are then the wrongranking. **Wyner–Ziv** tells us that when the decoder holds side information $Y$, therate needed is governed by the *conditional* quantity, not the marginal one — so what alatent handoff should preserve is what the **receiver cannot already recover**, not whatthe sender attended to.### The claim this notebook tries to break> At a fixed communication budget, ranking cache entries by **receiver-conditioned> surprisal** beats ranking them by sender-side salience *or* by sender-side surprisal,> and the margin **grows with the amount of side information the receiver holds**.The interaction — not the main effect — is the load-bearing prediction. A main effecthas many possible explanations; a margin that scales with the receiver's sideinformation is what *conditionality* specifically predicts.### What would falsify it| observation | verdict ||---|---|| sender-side rules match or beat receiver-conditioned ranking | claim is wrong || receiver-conditioned ranking ≈ literal deduplication of the sender ranking | the cross-field connection contributed nothing beyond "don't send duplicates" || the margin does not grow with side information | the *conditional* reading is not what is doing the work |Runtime: about 20 minutes on a Colab T4 with the default settings.

## Setup

In [ ]:
import os, sys, subprocessREPO = "https://github.com/SonghanWu/latent-message-handoff.git"ROOT = "latent-message-handoff"IN_COLAB = "google.colab" in sys.modulesif IN_COLAB:    os.chdir("/content")    if os.path.exists(ROOT):        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)    else:        subprocess.run(["git", "clone", "-q", REPO], check=True)    os.chdir(ROOT)    subprocess.run(        [sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"],        check=False,    )elif os.path.exists(ROOT):    os.chdir(ROOT)sys.path.insert(0, os.getcwd())# Fail loudly here rather than three cells later with a confusing ImportError.assert os.path.exists("latent_comm/__init__.py"), (    f"latent_comm not found under {os.getcwd()!r}. "    "This cell has to run (successfully) before every other cell in the notebook.")import torchprint("cwd:", os.getcwd())print("cuda:", torch.cuda.is_available(),      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from latent_comm.model_io import load_lmfrom latent_comm.kvcache import cache_bytes, scalars_per_token, to_legacyMODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # swap for Qwen2.5-1.5B-Instruct to check scalelm = load_lm(MODEL)cfg_model = lm.configprint(f"{MODEL}: {cfg_model.num_hidden_layers} layers, "      f"{cfg_model.num_attention_heads} query heads, {cfg_model.num_key_value_heads} KV heads, "      f"head_dim {cfg_model.hidden_size // cfg_model.num_attention_heads}")

## 0. The object under discussionBefore any argument, look at the thing we are proposing to compress. One forward passover a document, and the cache it leaves behind.

In [ ]:
from latent_comm.model_io import prefill_with_attentionprobe = lm.encode(" ".join(["the quick brown fox jumps over the lazy dog."] * 40))probe_cache, probe_attn = prefill_with_attention(lm, probe)k0, v0 = to_legacy(probe_cache)[0]n_tok = probe.shape[1]per_tok = scalars_per_token(probe_cache)print(f"document tokens          : {n_tok}")print(f"per-layer key tensor     : {tuple(k0.shape)}   (batch, kv_heads, seq, head_dim)")print(f"scalars per token        : {per_tok}  = 2 x {cfg_model.num_hidden_layers} layers "      f"x {cfg_model.num_key_value_heads} kv heads x {cfg_model.hidden_size // cfg_model.num_attention_heads}")print(f"bytes per token (fp16)   : {per_tok * 2:,}")print(f"whole cache              : {cache_bytes(probe_cache) / 1024:,.0f} KiB")print(f"at a 32k context         : {per_tok * 2 * 32768 / 1024**2:,.0f} MiB")

That last number is the project's premise: the cache grows linearly with every contextthe agent reads, so a full-state handoff is not an option and the interesting questionbecomes *which* part crosses.---## 1. The cross-field idea: coding with side information at the decoder**Source.** A. D. Wyner and J. Ziv, "The rate-distortion function for source coding withside information at the decoder," *IEEE Transactions on Information Theory*, 22(1):1–10,1976. ([DOI](https://doi.org/10.1109/TIT.1976.1055508)) Building on D. Slepian andJ. K. Wolf, "Noiseless coding of correlated information sources," *IEEE Transactions onInformation Theory*, 19(4):471–480, 1973.([DOI](https://doi.org/10.1109/TIT.1973.1055037))Slepian–Wolf established the counter-intuitive lossless result: if the decoder holdsside information $Y$, the encoder can compress $X$ to $H(X\mid Y)$ bits **even though theencoder never sees $Y$**. Wyner–Ziv extends this to lossy coding, characterising therate–distortion function $R_{WZ}(D)$ when $Y$ is available only at the decoder.Two consequences carry over to a latent handoff, and they are the only two I need:1. **What matters is conditional, not marginal.** The value of transmitting a piece of   $X$ is set by how much uncertainty about it remains *after* conditioning on $Y$.   Content the decoder can already reconstruct is worth nothing at any rate.2. **The distortion measure is a free choice.** Nothing in the theory says $d(x,\hat x)$   must be reconstruction error. Reconstruction is one admissible distortion among many;   a task-defined distortion is equally admissible — and for an agent handoff it is the   one that matters.Point 2 is what makes this more than a restatement of "don't send duplicates". It saysthe objective KV-compression methods optimise — faithfully reproducing the sender'sattention output — is a *choice*, and probably the wrong one when a downstream modelonly needs to answer a question.

## 2. Mapping the objects — and where the mapping stops| Wyner–Ziv | latent handoff | exact or analogy? ||---|---|---|| source $X$ | the worker's KV cache over the document | **exact** — a well-defined object we transmit || decoder side information $Y$ | the receiver's own context plus its model priors | **partly** — the context is exact; "model priors" is not a random variable we can condition on formally || message, rate $R$ | selected cache entries; scalars shipped | **exact** — countable, and we count them || distortion $d(x,\hat x)$ | receiver's NLL of the gold answer | **exact as a distortion**, but not the mean-squared error the theory's closed forms assume || encoder ignorant of $Y$ | H2O / SnapKV: ranking blind to the receiver | **exact** — this is the situation the theorem addresses |**Where it is only an analogy, stated plainly:*** A transformer's cache is not an i.i.d. source; there is no block length, no asymptotic  regime, and therefore no achievable-rate claim to inherit. All that survives is the  *direction* of the prediction.* We perform **selection**, not binning. Wyner–Ziv's remarkable part — that the encoder  reaches the conditional rate *without knowing* $Y$ — depends on a coset/binning  construction we do not implement. Our receiver-conditioned rule reads the receiver's  state directly, so it is an **oracle-side upper bound**, not a deployable codec. That  gap is the honest cost of the analogy and section 7 says what to do about it.* "Model priors" as side information is a gesture, not a formalism. We approximate it by  conditioning a forward pass on the receiver's context, which folds priors and context  together and cannot separate them.

## 3. The competing explanations, and the rules that instantiate themFive rules, one budget, same questions. Rules 3–5 differ *only* in what their scoringpass is conditioned on, which is what makes the comparison an isolation rather than ahorse race.| # | rule | explanation it stands for ||---|---|---|| 1 | `random` | floor || 2 | `sender_attention` | **salience**: what the sender attended to is what matters (H2O) || 3 | `sender_surprisal` | **information content**: task-conditioned, blind to the receiver || 4 | `dedup_sender_surprisal` | **surface redundancy**: same, minus what the receiver literally holds || 5 | `receiver_surprisal` | **conditional novelty**: scored inside the receiver's context |The decisive contrasts:* **5 vs 3** — does conditioning on the receiver help at all?* **5 vs 4** — does it help *beyond* dropping literal duplicates? Wyner–Ziv's redundancy  is what the decoder can *infer*, not only what it holds verbatim. If 5 ≈ 4, the  cross-field connection reduces to deduplication and I will say so.**Built-in invariant.** With zero side information, rules 3, 4 and 5 score with anidentical context and must select identical sets. The experiment asserts this; if itever fails, the two scoring passes are misaligned and every number is suspect.

## 4. Data: HotpotQA distractor as a side-information dial

In [ ]:
from latent_comm.data import load_examples, render_and_tokenize, choose_side_infoexamples = load_examples(n=100, seed=0)ex = examples[0]tex = render_and_tokenize(lm, ex)print(f"question : {ex.question}")print(f"answer   : {ex.answer}")print(f"paragraphs: {len(ex.paragraphs)}  (gold: {ex.gold_idx})")print(f"document : {tex.doc_len} tokens")print(f"full cache for this document: {tex.doc_len * scalars_per_token(probe_cache) * 2 / 1024:,.0f} KiB")

Each question ships with two gold paragraphs and eight distractors. We hand the receivera chosen number of **distractor** paragraphs as its own context and never a gold one, sothe evidence it needs always has to cross the handoff while the overlap between theworker's cache and the receiver's context is ours to set.

## 5. The three ranking signals, on one document

In [ ]:
from latent_comm.model_io import token_surprisalfrom latent_comm.experiment import render_side_infofrom latent_comm.stats import plot_scoresSIDE_LEVEL = 6side_paras = choose_side_info(tex, SIDE_LEVEL, seed=0)cache_full, attn_score = prefill_with_attention(lm, tex.doc_ids)sender_sup = token_surprisal(lm, lm.encode(f"Question: {ex.question}\n\n"), tex.doc_ids)receiver_sup = token_surprisal(    lm,    lm.encode(render_side_info(ex, side_paras) + f"Question: {ex.question}\n\n"),    tex.doc_ids,)os.makedirs("results", exist_ok=True)_ = plot_scores(tex, attn_score, sender_sup, receiver_sup, side_paras, "results/scores.png")print("receiver already holds paragraphs:", side_paras, " | gold:", ex.gold_idx)

Read the grey bands. Those are the paragraphs the receiver already has. Thereceiver-conditioned curve collapses over them — the model is copying from its owncontext, so nothing there is worth a byte — while the two sender-side curves carry on asif nothing were different, because nothing *is* different from where they are standing.That picture is the mechanism; the rest of the notebook asks whether it moves the metric.

## 6. The experiment* **Metric**: mean token NLL of the gold answer, read out of the handed-over cache.  Continuous, low-variance, and it needs no decoding — with a 0.5B model, exact-match on  free-form answers is mostly measuring formatting.* **Budget**: a percentage of the document's tokens, spent identically by every rule.  Positions the receiver already holds still cost budget when a rule picks them; that  waste is the phenomenon, not an accounting error.* **Position handling**: selected keys keep their original rotary phase and the question  is placed at `doc_len`. The receiver therefore sees a position sequence with holes —  the standard choice in this literature, and the one whose alternative (unrotate and  renumber) would destroy the original spacing between transmitted spans.* **Sinks**: every rule is forced to keep the first 4 positions, charged to its budget,  so attention sinks are not the hidden variable.* **Reference lines**: the full cache (upper bound on what any handoff can buy) and  side-information-only (what the receiver manages with no handoff at all).

In [ ]:
from latent_comm.experiment import Config, runfrom latent_comm.stats import to_framecfg = Config(    model_name=MODEL,    n_examples=len(examples),    side_info_levels=(0, 3, 6),    budget_fractions=(0.05, 0.10, 0.20),    seed=0,    out_dir="results",)rows = run(lm, examples, cfg)df = to_frame(rows)print(len(df), "rows")df.head()

## 7. Result

In [ ]:
from latent_comm.stats import main_figure_ = main_figure(df, "results/main.png")df.pivot_table(index="rule", columns="side_level", values="nll").round(4)

In [ ]:
from latent_comm.stats import comparison_tablecomparisons = comparison_table(df)comparisons.to_csv("results/comparisons.csv", index=False)comparisons.round(4)

Negative `delta_nll` means the first rule is better. `significant` is a paired bootstrapCI over questions that excludes zero — paired, because between-question variance is farlarger than the effect we are looking for.

In [ ]:
from latent_comm.stats import interaction_testinteraction = interaction_test(df, "receiver_surprisal", "dedup_sender_surprisal")interaction.to_csv("results/interaction.csv", index=False)interaction.round(4)

**This is the table that decides the claim.** `difference_of_differences` is how muchmore the receiver-conditioned rule wins when the receiver holds 6 distractor paragraphsthan when it holds none. The prediction is that it is negative with a CI clear of zero.If it straddles zero, the conditional reading is not carrying its weight, whatever themain effects say.

In [ ]:
# How much of each rule's budget went to content the receiver already had.df[df.side_level == 6].pivot_table(    index="rule", columns="budget_frac", values="wasted_budget").round(1)

## 8. Saved tensors

In [ ]:
from latent_comm.experiment import save_reference_tensorssave_reference_tensors(lm, examples[0], cfg, "results/reference_tensors.pt")blob = torch.load("results/reference_tensors.pt", weights_only=False)print("keys:", list(blob.keys()))print("layers saved:", blob["layer_ids"])print("key tensor shape per layer:", tuple(blob["kv"][0][0].shape))print("accumulated attention shape:", tuple(blob["accumulated_attention"].shape))print("\nresults/score_tensors.pt additionally holds, for the first 3 questions:")print("  accumulated attention, sender surprisal, receiver surprisal per side level,")print("  the side-information positions, and every rule's selected position set.")

## 9. Interpretation, alternatives, and where this breaks*Fill this in against the numbers above rather than from the hypothesis.***Does the result support the claim?** — state it against the interaction table, not themain effects.**One plausible alternative explanation.** The receiver-conditioned score is computed bya forward pass that has already read the side information, so it also inherits a*positional* bias: tokens near the end of the document get lower surprisal in everycondition, and the amount of that varies with prefix length. A control that matchesprefix *length* while scrambling its *content* (shuffled paragraphs from otherquestions) would separate "conditioned on this receiver's context" from "conditioned ona longer prefix". That control is the first thing I would add.**Where the cross-field connection breaks down.** Our rule is not a Wyner–Ziv code. Thetheorem's substance is that the encoder can reach the conditional rate *without seeing*$Y$, via binning; we instead let the scorer read $Y$ directly. So this experimentmeasures the value of the conditional *objective*, and says nothing about whether it isreachable under the constraint that makes Wyner–Ziv interesting. A deployable versionneeds either a compact digest of the receiver's state sent upstream (which costs rateand should be charged to the budget) or a genuine binning scheme.**Follow-up experiment.** Charge the receiver→sender digest to the same budget and seewhether the advantage survives paying for it. If a digest of $k$ scalars recovers mostof the oracle gap while costing far less than the KV entries it saves, the idea ispractical; if not, the oracle result is a curiosity.